
# ✅ S11 - Soluciones de Supervised Learning
Soluciones oficiales de los ejercicios del módulo.


In [ ]:

import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report
)


## 🚢 Carga del Dataset

In [ ]:

df = sns.load_dataset("titanic")
df = df[["survived", "pclass", "sex", "age", "fare", "embarked"]]
df = df.dropna()


## ✅ Exploración

In [ ]:

df.head()
df.shape
df.dtypes
df.isnull().sum()


## ✅ One-Hot Encoding

In [ ]:

ohe = OneHotEncoder(sparse_output=False)
encoded = ohe.fit_transform(df[["sex", "embarked"]])
encoded_df = pd.DataFrame(
    encoded,
    columns=ohe.get_feature_names_out(["sex", "embarked"])
)
encoded_df.head()


## ✅ Ordinal Encoding

In [ ]:

ordinal = OrdinalEncoder()
df["pclass_encoded"] = ordinal.fit_transform(df[["pclass"]])
df.head()


## ✅ Escalamiento

In [ ]:

scaler_std = StandardScaler()
scaler_mm = MinMaxScaler()

df[["age_std", "fare_std"]] = scaler_std.fit_transform(df[["age", "fare"]])
df[["age_mm", "fare_mm"]] = scaler_mm.fit_transform(df[["age", "fare"]])


## ✅ División de los Datos

In [ ]:

X = df.drop("survived", axis=1)
y = df["survived"]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


## ✅ Regresión Logística

In [ ]:

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

accuracy_score(y_test, y_pred)


## ✅ Métricas

In [ ]:

precision_score(y_test, y_pred)
recall_score(y_test, y_pred)
f1_score(y_test, y_pred)
print(classification_report(y_test, y_pred))


## ✅ Datos Desbalanceados

In [ ]:

model_balanced = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)

print(classification_report(y_test, y_pred_balanced))


## ✅ ColumnTransformer y Pipeline

In [ ]:

num_cols = ["age", "fare"]
cat_cols = ["sex", "embarked", "pclass"]

X = df.drop("survived", axis=1)
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first"), cat_cols)
    ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000))
    ]
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

accuracy_score(y_test, y_pred)


## ✅ Nombres de las Columnas Transformadas

In [ ]:

feature_names = pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

feature_names


## 🌟 Bonus: Dataset Tips

In [ ]:

tips = sns.load_dataset("tips")

X = tips[["total_bill", "size"]]
y = tips["tip"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
model.score(X_test, y_test)
